# Day 5: ANSYS FEA Result File Parser & Automated Deflection Inspector
**Author:** Navneet Namdeo  
**Domain:** FEA Automation / CAE Post-Processing  
**Repository:** Python-Automation-Projects

### Overview
This notebook creates a synthetic ANSYS nodal displacement output file (`.csv`) and builds an automated parsing pipeline using Pandas. It filters nodes exceeding maximum allowable deformation limits ($\delta_{max}$) and generates an automated structural safety report.

In [1]:
import pandas as pd
import numpy as np

# 1. Generate Synthetic ANSYS FEA Export File (Nodal Displacement Table)
np.random.seed(101)
num_nodes = 500

node_ids = np.arange(1, num_nodes + 1)
x_coords = np.random.uniform(0, 1000, num_nodes)  # mm
y_coords = np.random.uniform(0, 100, num_nodes)   # mm
z_coords = np.random.uniform(0, 50, num_nodes)    # mm

# Simulate displacement results (U_total in mm)
ux = np.random.normal(0, 0.05, num_nodes)
uy = np.random.normal(-0.25, 0.15, num_nodes)  # Primary load direction
uz = np.random.normal(0, 0.02, num_nodes)

# Compute resultant displacement U_sum = sqrt(Ux^2 + Uy^2 + Uz^2)
u_sum = np.sqrt(ux**2 + uy**2 + uz**2)

ansys_export_df = pd.DataFrame({
    'Node_ID': node_ids,
    'X_mm': np.round(x_coords, 2),
    'Y_mm': np.round(y_coords, 2),
    'Z_mm': np.round(z_coords, 2),
    'Ux_mm': np.round(ux, 4),
    'Uy_mm': np.round(uy, 4),
    'Uz_mm': np.round(uz, 4),
    'U_sum_mm': np.round(u_sum, 4)
})

# Save to local directory to simulate real ANSYS CSV export
ansys_export_df.to_csv('ansys_nodal_displacements.csv', index=False)
print("--> Simulated ANSYS export file saved as 'ansys_nodal_displacements.csv'\n")

# 2. Automated FEA Deflection Inspector Engine
def inspect_fea_deflection(file_path, max_allowable_deflection_mm=0.45):
    """
    Parses ANSYS nodal displacement CSV file, identifies critical nodes,
    and outputs structural compliance metrics.
    """
    # Load FEA result file
    df = pd.read_csv(file_path)
    
    # Calculate global max deflection
    max_disp_row = df.loc[df['U_sum_mm'].idxmax()]
    max_disp_value = max_disp_row['U_sum_mm']
    critical_node_id = int(max_disp_row['Node_ID'])
    
    # Filter nodes exceeding allowable deflection threshold
    failed_nodes = df[df['U_sum_mm'] > max_allowable_deflection_mm].sort_values(by='U_sum_mm', ascending=False)
    
    # Summary Report
    print("=" * 65)
    print("           ANSYS FEA AUTOMATED POST-PROCESSING REPORT")
    print("=" * 65)
    print(f"Total Nodes Analyzed          : {len(df)}")
    print(f"Allowable Deflection Limit    : {max_allowable_deflection_mm:.3f} mm")
    print(f"Maximum Deflection Observed   : {max_disp_value:.3f} mm (At Node #{critical_node_id})")
    print("-" * 65)
    
    if len(failed_nodes) == 0:
        print("COMPLIANCE STATUS: PASSED (All nodes within deflection limits)")
    else:
        print(f"COMPLIANCE STATUS: WARNING ({len(failed_nodes)} nodes exceed allowable limit!)")
        print("\nTOP 5 CRITICAL OVER-DEFLECTED NODES:")
        print(failed_nodes[['Node_ID', 'X_mm', 'Y_mm', 'Z_mm', 'U_sum_mm']].head().to_string(index=False))
    print("=" * 65)
    
    # Save flagged critical nodes to CSV for FEA engineer review
    if len(failed_nodes) > 0:
        failed_nodes.to_csv('critical_overdeflected_nodes.csv', index=False)
        print("\n--> Critical nodes exported to 'critical_overdeflected_nodes.csv'")

# Run Inspector with allowable deflection limit = 0.45 mm
inspect_fea_deflection('ansys_nodal_displacements.csv', max_allowable_deflection_mm=0.45)

--> Simulated ANSYS export file saved as 'ansys_nodal_displacements.csv'

           ANSYS FEA AUTOMATED POST-PROCESSING REPORT
Total Nodes Analyzed          : 500
Allowable Deflection Limit    : 0.450 mm
Maximum Deflection Observed   : 0.847 mm (At Node #142)
-----------------------------------------------------------------
COMPLIANCE STATUS: WARNING (37 nodes exceed allowable limit!)

TOP 5 CRITICAL OVER-DEFLECTED NODES:
 Node_ID   X_mm  Y_mm  Z_mm  U_sum_mm
     142 100.85 97.61 40.03    0.8472
     286 137.38 90.31  9.84    0.6562
     474 980.56 95.68 19.20    0.6317
     156 580.92 98.33 45.27    0.6298
     368  44.05 34.38 44.80    0.6037

--> Critical nodes exported to 'critical_overdeflected_nodes.csv'
